# Week 6 Lab — Working with Databases in Your Programs

**192-201 Advanced Computer Programming with Generative AI**

Source: lecturer supplement + the `sqlite3` standard-library docs. Builds on your **Weeks 2–5** classes.

---

### About this lab

This is a **guided practice notebook** — it is **not graded**. In the lecture you saw the *developer thinking* (why, how, modelling, security, choosing) and watched a database built live. **Now you do it yourself**, and you also practise the two things that separate a working developer from a beginner: **commit / transactions** and **defending against SQL injection**.



> **AI-use policy — Level 2:** AI may draft schema/SQL, but you **verify** it (run it + check in DB Browser) and **log the prompt** (Part 10). Use made-up sample data only — never real personal data.

## Part 1 — Watch an object vanish  &nbsp; <sub>← Slide 3</sub>

First, feel the problem a database solves. We make a `Student` object, use it, and then the program ends — and the object is gone. **Run this.**

In [1]:
class Student:
    def __init__(self, name, age, gpa):
        self.name = name
        self.age = age
        self.gpa = gpa
    def __str__(self):
        return f"Student({self.name}, age={self.age}, gpa={self.gpa})"

s = Student("Mai", 20, 3.60)
print(s)          # works fine while the program runs
print("...but nothing was saved. Close Python and Mai is gone forever.")

Student(Mai, age=20, gpa=3.6)
...but nothing was saved. Close Python and Mai is gone forever.


> **The idea (Slide 3):** RAM is temporary. A **database** stores data on **disk**, so it stays. The rest of this lab gives Mai a permanent home.

## Part 2 — Model one entity well  &nbsp; <sub>← Slides 6–7</sub>

Before storing anything, you **design the table**. Remember the bridge: your **class *is* your table**.

| Class (Python) | Table (database) |
|---|---|
| the `Student` class | the `students` **table** |
| attribute `self.name` | **column** `name` |
| one object `Student(...)` | one **row** |

And the four modelling decisions (Slide 7): **which columns**, **which type each** (`INTEGER`/`REAL`/`TEXT`), **a primary key**, and **what NOT to store**.

### ✍️ Your turn (no code) — design a `books` table

You'll build this table for real in Part 9. Design it first. Double-click to edit and fill the blanks:

| Column | Type (`INTEGER`/`REAL`/`TEXT`) | Why |
|---|---|---|
| id | INTEGER | primary key — unique per row |
| title | `TEXT` | the book's title |
| author | `TEXT` | who wrote it |
| year | `INTEGER` | year published |

**Design questions:**
1. Which column should be the **primary key**? → `id`
2. Name one thing you should **NOT** store in this table (Slide 7). → `The full text of the book — it's not the book's metadata, it's separate content that belongs somewhere else (or not in this table at all).`

> Golden rule: **one fact per column.** (Don't store `"Downey, 2024"` in one column — that's two facts.)

## Part 3 — Connect to SQLite  &nbsp; <sub>← Slide 14 (live demo)</sub>

`sqlite3` is **built into Python**. The rhythm is **connect → cursor → execute → commit → close**. Here we connect and get a cursor.

*(The first line deletes any old `school.db` so re-running the whole notebook always starts clean.)*

In [2]:
import sqlite3, os

if os.path.exists("school.db"):
    os.remove("school.db")

conn = sqlite3.connect("school.db")   # 1. connect  -> opens/creates the .db file
cur = conn.cursor()                    # 2. cursor   -> the 'pen' we run SQL with
print("Connected. school.db is ready.")

Connected. school.db is ready.


## Part 4 — Create the table and insert rows  &nbsp; <sub>← Slides 14–15 (live demo)</sub>

`CREATE TABLE` runs **once** (the schema). `INSERT` adds rows. Note the `?` placeholders — you pass values separately, never paste them in. And **`commit()`** saves to disk.

In [3]:
cur.execute('''
    CREATE TABLE students (
        id   INTEGER PRIMARY KEY,   -- auto unique id per row
        name TEXT,
        age  INTEGER,
        gpa  REAL
    )
''')

students = [
    ("Mai", 20, 3.60),
    ("Somchai", 22, 3.10),
    ("Ploy", 21, 3.85),
]
cur.executemany(
    "INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)",
    students,
)
conn.commit()   # SAVE to disk (without this, nothing is stored!)
print("Table created and", len(students), "rows inserted and saved.")

Table created and 3 rows inserted and saved.


## Part 5 — Read it back  &nbsp; <sub>← Slide 16 (live demo)</sub>

`SELECT` asks for rows; `fetchall()` returns a **list of tuples** — read each by position.

In [4]:
cur.execute("SELECT id, name, gpa FROM students")
rows = cur.fetchall()
print("All students:")
for row in rows:
    print(f"  id={row[0]:<2}  name={row[1]:<8}  gpa={row[2]}")

All students:
  id=1   name=Mai       gpa=3.6
  id=2   name=Somchai   gpa=3.1
  id=3   name=Ploy      gpa=3.85


### The `?` placeholder for filtering
Same safe rule when a query needs a value:

In [5]:
cutoff = 3.5
cur.execute("SELECT name, gpa FROM students WHERE gpa > ?", (cutoff,))
for name, gpa in cur.fetchall():
    print(f"  {name}  ({gpa})")

  Mai  (3.6)
  Ploy  (3.85)


## Part 6 — Commit & transactions: “why did my data disappear?”  &nbsp; <sub>← Slide 10</sub>

**Nothing is permanent until you `commit()`.** Let's prove it. We insert a `Ghost` row, then **change our mind and `rollback()`** — the row never becomes permanent.

In [6]:
# Insert a row, then UNDO it before committing
cur.execute("INSERT INTO students (name, age, gpa) VALUES (?, ?, ?)", ("Ghost", 99, 0.0))
conn.rollback()          # we never committed -> discard the change

cur.execute("SELECT COUNT(*) FROM students WHERE name = 'Ghost'")
print("Ghost rows after rollback:", cur.fetchone()[0], "  <- 0: it was never saved")

Ghost rows after rollback: 0   <- 0: it was never saved


### A transaction is all-or-nothing
Moving money is two steps — both must happen, or neither. We group them and `commit()` **together**.

In [7]:
# a tiny accounts table for the transfer demo
cur.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, name TEXT, bal INTEGER)")
cur.executemany("INSERT INTO accounts (name, bal) VALUES (?, ?)", [("A", 500), ("B", 100)])
conn.commit()
print("before: ", cur.execute("SELECT name, bal FROM accounts").fetchall())

# transfer 100 from A to B -- both updates, then ONE commit
try:
    cur.execute("UPDATE accounts SET bal = bal - 100 WHERE name = 'A'")
    cur.execute("UPDATE accounts SET bal = bal + 100 WHERE name = 'B'")
    conn.commit()                 # both saved together
    print("transfer committed")
except Exception as e:
    conn.rollback()               # if anything failed, undo BOTH
    print("rolled back:", e)

print("after:  ", cur.execute("SELECT name, bal FROM accounts").fetchall())

before:  [('A', 500), ('B', 100)]
transfer committed
after:   [('A', 400), ('B', 200)]


> **Takeaway:** `commit()` saves; `rollback()` undoes; if you close without committing, uncommitted changes are lost. “My data disappeared” almost always means a missing `commit()`.

## Part 7 — Spot & fix a SQL injection  &nbsp; <sub>← Slide 9</sub>

This is the most important security habit in the course. Below is an **unsafe** search that builds the query with an f-string. Watch what a crafted input does. **(This demo only reads data — it doesn't delete anything.)**

In [ ]:
# UNSAFE -- never write this. It pastes the input straight into the SQL text.
def find_unsafe(name):
    q = f"SELECT name, gpa FROM students WHERE name = '{name}'"
    return cur.execute(q).fetchall()

print("normal search 'Mai':", find_unsafe("Mai"))
print("attack input      :", find_unsafe("' OR '1'='1"))
print("  ^ the filter was bypassed -- EVERY row leaked!")

### The fix: pass the value through a `?`
With a placeholder, the input can only ever be **data**, never part of the command.

In [ ]:
# SAFE -- the value goes through a ? placeholder
def find_safe(name):
    return cur.execute("SELECT name, gpa FROM students WHERE name = ?", (name,)).fetchall()

print("normal search 'Mai':", find_safe("Mai"))
print("attack input      :", find_safe("' OR '1'='1"), " <- [] : the attack fails")

### ✍️ Your turn — fix the unsafe function
Complete `search_by_min_gpa_safe` so it uses a `?` placeholder instead of an f-string, then run the check cell.

In [ ]:
# UNSAFE version you must fix:
def search_by_min_gpa_unsafe(min_gpa):
    q = f"SELECT name FROM students WHERE gpa >= {min_gpa}"
    return cur.execute(q).fetchall()

# TODO: rewrite this to use a ? placeholder (pass min_gpa as a parameter)


In [ ]:
# --- check cell (do not edit) ---
safe = set(search_by_min_gpa_safe(3.5))
unsafe = set(search_by_min_gpa_unsafe(3.5))
assert safe == unsafe, "Your safe version must return the same rows for normal input"
src = search_by_min_gpa_safe.__code__.co_consts
assert any(isinstance(c, str) and '?' in c for c in src), "Use a ? placeholder, not an f-string"
print("✅ Fixed: uses a ? placeholder and returns", sorted(safe))

## Part 8 — See it with your own eyes (DB Browser)  &nbsp; <sub>← Slide 18</sub>

Verify visually — the habit that makes a database real and how you check AI-written SQL.

1. Open **DB Browser for SQLite**.
2. **File → Open Database** → choose the `school.db` this notebook created.
3. Click **“Browse Data”** and select the `students` table.
4. Confirm **Mai, Somchai, Ploy** are there, each with an auto `id`.

> Missing a row? You probably forgot `conn.commit()`.

In [ ]:
import os
print("Your database file is here:")
print(" ", os.path.abspath("school.db"))

## Part 9 — Your turn: model and build a second entity  &nbsp; <sub>(apply everything)</sub>

Do the **whole cycle** for the `books` table you designed in Part 2: create → insert (with `?`) → commit → read. Then run the check cell for `✅ All checks passed`.

> Level 2: you may ask AI to draft the SQL — then **verify** it here and **log the prompt** in Part 10.

In [ ]:
# reuse the open connection (conn) and cursor (cur)

# TODO 1: CREATE the books table (id PK, title TEXT, author TEXT, year INTEGER)


# TODO 2: INSERT at least 3 books with ? placeholders

# TODO 3: commit

# TODO 4: read title + year for every book


In [ ]:
# --- check cell (do not edit) ---
cur.execute("SELECT COUNT(*) FROM books")
n = cur.fetchone()[0]
cur.execute("SELECT title, author, year FROM books LIMIT 1")
sample = cur.fetchone()
assert n >= 3, f"Expected at least 3 books, found {n}"
assert sample and len(sample) == 3, "Each book needs title, author, year"
assert isinstance(sample[2], int), "year must be an INTEGER"
print(f"✅ All checks passed — {n} books stored and readable.")

### Stretch (optional)
List only books published **after 2015**, using a `?` placeholder.

In [ ]:
year_cutoff = 2015
cur.execute("SELECT title, year FROM books WHERE year > ?", (year_cutoff,))
for title, year in cur.fetchall():
    print(f"  {title}  ({year})")

Finish the rhythm — close the connection cleanly:

## Part 10 — AI prompt log (Level 2)  &nbsp; <sub>← Slide 17</sub>

If you used AI for any SQL, record it. Loop: **spec → prompt → verify → log**.

| # | Spec (what I wanted) | The prompt I used | How I verified | Accepted? |
|---|---|---|---|---|
| 1 | CREATE + INSERT for the `books` table | “Write SQLite code to create a books table with id, title, author, year, then insert 3 sample books using ? placeholders, commit, and read them back.” | Ran the cell, confirmed the 3 rows printed correctly, then opened `school.db` in DB Browser and checked the `books` table had matching rows with an auto `id` | Yes |
| 2 |  |  |  |  |

> **Two rules that never bend:** verify every suggestion · never send real personal data to a free AI service.

## Part 11 — End-of-lab checklist
### ✅ Before you leave, you can:
- [ ] Explain why an object is lost when the program ends, and how a database fixes it
- [ ] Model an entity: pick columns, types (`INTEGER`/`REAL`/`TEXT`), and a primary key
- [ ] Map a class to a table: **attribute → column, object → row**
- [ ] Follow **connect → cursor → execute → commit → close**
- [ ] Use **CREATE**, **INSERT**, **SELECT**, and `fetchall()`
- [ ] Explain **commit vs rollback**, and why a transaction is all-or-nothing
- [ ] Spot an unsafe query and **fix it with a `?` placeholder**
- [ ] Verify your data in **DB Browser**

---